# 01 - Baseline Evaluation

Establishes baseline performance for handwriting recognition models:
- Tesseract 5 OCR
- TrOCR pretrained (no fine-tuning)
- VLM zero-shot

Tested on IAM, GNHK, and SMHD test sets.

In [ ]:
# Install dependencies
# !pip install transformers evaluate jiwer torch torchvision pillow

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import evaluate

# Load pretrained model
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')
model.eval()

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device: {next(model.parameters()).device}')

In [ ]:
# Inference example
def predict_single(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt').pixel_values
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=128,
            num_beams=4,
        )
    
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text

# Example usage:
# result = predict_single('path/to/handwriting_line.png')
# print(f'Predicted: {result}')

## Evaluation Metrics

Key metrics:
- **CER** (Character Error Rate): Primary metric
- **WER** (Word Error Rate): Secondary metric
- **Calibration Error**: How well confidence matches accuracy
- **False Confidence Rate**: High-confidence errors